# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the FAIR² (FAIR2) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`


In [ ]:
# Ensure `mlcroissant` is installed in this environment
!pip install mlcroissant

## 1. Data Loading
Load Croissant dataset metadata and prepare the access to records.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset metadata from the schema
dataset = mlc.Dataset(croissant_url)

# Print the dataset title and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review all available Record Sets and their corresponding Fields by `@id`. Each entity in Croissant (RecordSets, Fields, Columns) can be uniquely referenced by its `@id`.

In [ ]:
from collections import defaultdict

# List all record sets by @id and their fields
record_sets = list(dataset.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}) [type: {field.data_type}]")
    print("")

## 3. Data Extraction
Load data from each Record Set into a DataFrame for analysis.

Use Record Set and Field `@id`s from the previous step for reference.

In [ ]:
# Prepare to extract records for all record sets
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

# For demonstration, load first 5 rows from each record set
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(dataframes[record_set_id])} records for RecordSet: {record_set_id}")
    else:
        print(f"No records found for RecordSet: {record_set_id}")

# Show columns of the first available record set with data
for rsid, df in dataframes.items():
    print(f"\nColumns for RecordSet (@id={rsid}):\n{df.columns.tolist()}")
    display(df.head())
    break  # Show only the first available dataset

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps like filtering, normalizing numeric fields, or grouping. Reference fields using their `@id` only.

- Select a numeric field by its exact `@id`. (Replace the example with actual field `@id` from the overview, as needed. Adjust the logic for your selected field and record set.)

In [ ]:
# Example: Pick the first record set with data and select a numeric field for demonstration
selected_record_set = None
numeric_field_id = None
group_field_id = None

# Attempt to auto-detect a numeric field (Float or Integer type)
for rs in dataset.record_sets:
    df = dataframes.get(rs.id)
    if df is not None and not df.empty:
        for field in rs.fields:
            if field.data_type in ["Float", "Integer"] and field.id in df.columns:
                selected_record_set = rs.id
                numeric_field_id = field.id
                break
        # Try to find a group field (categorical/text) for grouping
        if selected_record_set:
            for field in rs.fields:
                if field.data_type in ["Text", "String"] and field.id in df.columns:
                    group_field_id = field.id
                    break
        if selected_record_set and numeric_field_id:
            break

if not selected_record_set:
    print("No suitable RecordSet and numeric field found for EDA.")
else:
    print(f"Using RecordSet: {selected_record_set}")
    print(f"Numeric field: {numeric_field_id}")
    df = dataframes[selected_record_set]

    # Drop rows where the numeric field is missing
    filtered_df = df.dropna(subset=[numeric_field_id])

    # For demonstration, filter values greater than the mean
    mean_val = filtered_df[numeric_field_id].astype(float).mean()
    filter_threshold = mean_val  # Set threshold to mean
    filtered_df = filtered_df[filtered_df[numeric_field_id].astype(float) > filter_threshold]
    print(f"Filtered records with {numeric_field_id} > {filter_threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the numeric field (Z-score)
    col = numeric_field_id
    filtered_df[f"{col}_normalized"] = (filtered_df[col].astype(float) - filtered_df[col].astype(float).mean()) / filtered_df[col].astype(float).std()
    print(f"Normalized {col} for filtered records:")
    display(filtered_df[[col, f"{col}_normalized"]].head())

    # Group by group_field_id if available
    if group_field_id and group_field_id in filtered_df.columns:
        print(f"\nGrouping by {group_field_id}:\n")
        group_means = filtered_df.groupby(group_field_id)[col].mean().to_frame("mean").head()
        display(group_means)
    else:
        print("No suitable group field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

- Example: Histogram of the numeric field. Modify as needed for your data exploration.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set and numeric_field_id and not filtered_df.empty:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field_id].astype(float), kde=True, bins=20)
    plt.title(f"Distribution of '{numeric_field_id}' in RecordSet: {selected_record_set}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()
    
    if group_field_id and group_field_id in filtered_df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=filtered_df[group_field_id], y=filtered_df[numeric_field_id].astype(float))
        plt.title(f"'{numeric_field_id}' by '{group_field_id}'")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to:
- Load a FAIR² dataset with Croissant schema using `mlcroissant`.
- Programmatically examine Record Sets, Fields, and data by their unique `@id`s.
- Extract, filter, normalize, and visualize real-world survey results using Pandas and Seaborn.

Adjust the exploration steps to focus on variables and Record Sets of interest for your analysis of rangeland management practices and adoption predictors in Northern Kenya.